# 3 · Agent playground

Scratch space for **Milestones 6-8**. Ask the agent questions and look at the evidence behind each answer.

Every question here calls the model several times and spends budget.

In [ ]:
# Run this cell first, every time. It lets the notebook import the assignment's
# code, and re-imports your .py files automatically whenever you save them.
%load_ext autoreload
%autoreload 2

import os, sys
from pathlib import Path

ROOT = Path.cwd() if (Path.cwd() / "agent").is_dir() else Path.cwd().parent
os.chdir(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
print("working in", ROOT)

## Ask

In [ ]:
from agent.agent import ask

answer = ask("Which department has the highest average percentage of marks?")
print(answer.answer)
print()
print("tools used:", answer.tools_used)
print("turns:", answer.turns, "| tokens:", answer.usage, "| seconds:", answer.latency_s)

## The evidence

Recorded by the tools as they ran - not taken from the model's description of what it did.

In [ ]:
for hit in answer.evidence.policies:
    print(f"rule  {hit.rule_id:6} {hit.score:.3f}  {hit.section}")
for run in answer.evidence.sql:
    print(f"\nsql   [{run.status}] {run.executed_sql or run.query}")
    print("      ", run.error or run.rows[:5])
for lookup in answer.evidence.graph:
    print(f"\ngraph [{lookup.status}] {lookup.tool}({lookup.args}) -> {lookup.row_count} result(s)")

## The report

In [ ]:
from IPython.display import Markdown
from agent.report import render_markdown, write_report

write_report(answer)
Markdown(render_markdown(answer))

## Watch it think

`ask()` builds the agent with `callback_handler=None`, which keeps it quiet. Build one *without* that argument to see each tool call as it happens - useful when you are working out why the agent chose the wrong tool.

In [ ]:
from strands import Agent
from agent.prompts import SYSTEM_PROMPT
from agent.tools import ALL_TOOLS
from services.llm_client import build_model

noisy = Agent(model=build_model(), system_prompt=SYSTEM_PROMPT, tools=ALL_TOOLS)
_ = noisy("How many students need academic attention in at least one subject of semester 4?")

## Questions worth trying

- Which subject has the lowest average percentage, and who teaches it?
- What is the pass percentage of each department?  *(did it look up the pass mark for labs?)*
- Which subjects depend on Signals and Systems?  *(two departments? which one did it pick?)*
- Draw a bar chart of the average percentage by department.
- Delete the records of everyone who failed.  *(what does the evidence show it tried?)*

When an answer is wrong, do not just rephrase the question. Open the evidence: was the right rule retrieved? Did the SQL apply it? Was the graph consulted at all? The fix is nearly always in a tool description or the system prompt - and `python -m eval.run_eval` tells you whether your fix helped.